# 05 — Tinh chỉnh prompt tiếng Việt (Gia Thoại, N3)

**Không cần GPU.** Notebook này là vòng lặp: sửa prompt → chấm điểm → xem câu sai → sửa tiếp.

**Quan trọng — hiểu đúng việc mình đang làm:**
- Claude **không train được** (Anthropic không cho fine-tune). Cách duy nhất để nó làm tốt hơn là **viết hướng dẫn + ví dụ mẫu tốt hơn** — đó chính là prompt engineering, và đó là việc của bạn.
- Claude **đã hiểu tiếng Việt rất tốt**. Thứ không hiểu tiếng Việt là **CLIP** (model tìm ảnh). Vai trò của bạn là làm "người phiên dịch": biến tiếng Việt mơ hồ → mô tả thị giác tiếng Anh mà CLIP hiểu.
- Thước đo duy nhất: **điểm trên bộ test**. Sửa prompt xong phải chạy lại, không đoán.

In [ ]:
# === BƯỚC 1: Cài thư viện + lấy code của đội ===
%pip -q install anthropic

import os, subprocess
if not os.path.isdir('/content/aic2026'):
    # Repo private: Colab sẽ hỏi đăng nhập GitHub (File > Open notebook > GitHub đã làm việc này)
    print('Đang tải code đội...')
    subprocess.run(['git', 'clone',
                    'https://github.com/dungnguyen02102007-commits/aic2026.git',
                    '/content/aic2026'], check=False)
%cd /content/aic2026
!git checkout dung/tuan1-khoi-dong-3-mang 2>/dev/null; git pull --quiet
print('OK — code sẵn sàng')

In [ ]:
# === BƯỚC 2: Cắm API key AN TOÀN (dùng Colab Secrets, KHÔNG gõ key vào ô code) ===
# Cách làm: nhìn thanh bên trái Colab -> biểu tượng CHÌA KHÓA 🔑 -> + Add new secret
#   Name  = ANTHROPIC_API_KEY
#   Value = key của bạn (sk-ant-api03-...)
#   Bật công tắc "Notebook access"
# Key nằm trong tài khoản Google của bạn, KHÔNG bao giờ nằm trong file notebook.

from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
os.environ['AIC_LLM_PROVIDER'] = 'anthropic'
os.environ['AIC_LLM_MODEL'] = 'claude-sonnet-5'   # đổi sang claude-haiku-4-5 để rẻ hơn khi thử nhiều

import anthropic
r = anthropic.Anthropic().messages.create(
    model=os.environ['AIC_LLM_MODEL'], max_tokens=32,
    messages=[{'role': 'user', 'content': 'Trả lời đúng 2 chữ: sẵn sàng'}])
print('Key hoạt động:', next(b.text for b in r.content if b.type == 'text'))

In [ ]:
# === BƯỚC 3: Chấm điểm hiện tại (chuẩn để so sánh) ===
!python -m agent.test_understand

## Bước 4 — Vòng lặp tinh chỉnh

Toàn bộ "bộ não" của người phiên dịch nằm trong `agent/prompts.py`, gồm 2 phần:
- `UNDERSTAND_SYSTEM` — bản hướng dẫn (viết tiếng Việt được)
- `UNDERSTAND_FEWSHOT` — các ví dụ mẫu (dạy bằng ví dụ hiệu quả hơn dạy bằng lời)

Cell dưới cho phép sửa prompt **ngay trong Colab** rồi chấm lại — không cần sửa file.

In [ ]:
# === Sửa prompt tại chỗ rồi chấm lại ===
import importlib, agent.prompts, agent.understand, agent.test_understand
importlib.reload(agent.prompts)

# ---- SỬA Ở ĐÂY: thêm luật mới vào cuối bản hướng dẫn ----
LUAT_THEM = """
7. Người dùng có thể gõ KHÔNG DẤU (vd: "tim canh nguoi dan ong doi mu").
   Hãy tự hiểu và xử lý như khi có dấu đầy đủ.
8. Từ địa phương: "ly"="cốc", "xe hơi"="ô tô", "bông"="hoa", "chén"="bát",
   "heo"="lợn", "dĩa"="đĩa". Đừng dịch sai vì khác vùng miền.
9. Từ đa nghĩa phải chọn theo ngữ cảnh: "con chuột" trong nhà bếp là động vật,
   cạnh máy tính là thiết bị. "đá" có thể là hòn đá, nước đá, hoặc hành động đá.
"""

agent.prompts.UNDERSTAND_SYSTEM = agent.prompts.UNDERSTAND_SYSTEM + LUAT_THEM
importlib.reload(agent.understand)
agent.understand.UNDERSTAND_SYSTEM = agent.prompts.UNDERSTAND_SYSTEM

importlib.reload(agent.test_understand)
agent.test_understand.main()

In [ ]:
# === BƯỚC 5: Bài kiểm tra RIÊNG cho tiếng Việt (cạm bẫy thật lúc thi) ===
# Người thi gõ rất nhanh: bỏ dấu, viết tắt, dùng từ địa phương.
# Nếu người phiên dịch chết ở đây thì lúc thi là mất câu.
import json
from agent.understand import understand

BAY_TIENG_VIET = [
    ('KHÔNG DẤU',    'tim canh nguoi dan ong doi mu di qua duong'),
    ('KHÔNG DẤU',    'canh an kem ben bo bien luc hoang hon'),
    ('TỪ MIỀN NAM',  'Tìm cảnh cô gái cầm ly cà phê đứng cạnh chiếc xe hơi màu đỏ'),
    ('TỪ MIỀN BẮC',  'Tìm cảnh cô gái cầm cốc cà phê đứng cạnh chiếc ô tô màu đỏ'),
    ('ĐA NGHĨA',     'Tìm cảnh con chuột chạy trong bếp'),
    ('ĐA NGHĨA',     'Tìm cảnh người ta đá bóng trên bãi đá'),
    ('VIẾT TẮT',     'tìm cảnh 2 ng đag ăn cơm ở qán vỉa hè'),
    ('TỪ MƯỢN',      'Cảnh mọi người đi sốp-ping ở trung tâm thương mại'),
    ('VĂN NÓI',      'cho tui coi cái đoạn mà ổng đang sửa xe máy đó'),
    ('TRỪU TƯỢNG',   'Tìm cảnh không khí ấm cúng của gia đình vào buổi tối'),
]

for nhom, cau in BAY_TIENG_VIET:
    try:
        r = understand(cau)
        print(f"[{nhom}] {cau}")
        print(f"   -> {r['search_text_en']}")
        print(f"      loại: {r['task_type']} | positives: {r['positives']}\n")
    except Exception as e:
        print(f"[{nhom}] LỖI: {e}\n")

# ĐỌC KẾT QUẢ: bản dịch tiếng Anh có mô tả ĐÚNG cảnh không?
# - 'ly cà phê' và 'cốc cà phê' phải ra CÙNG một mô tả tiếng Anh
# - 'con chuột trong bếp' phải là con vật (mouse/rat), KHÔNG phải computer mouse
# - câu không dấu phải hiểu đúng như câu có dấu
# Chỗ nào sai -> thêm luật/ví dụ vào prompts.py -> chạy lại BƯỚC 3 xem điểm có tụt không

In [ ]:
# === BƯỚC 6: Lưu prompt tốt nhất về máy để đưa vào repo ===
# Sau khi tìm được bản prompt cho điểm cao nhất, copy nội dung ra và dán vào
# agent/prompts.py trên máy, rồi commit qua Pull Request.
print(agent.prompts.UNDERSTAND_SYSTEM)